# Central Virginia Tree Canopy — Policy Modeling Data Pipeline

**Purpose:** Download and merge four external policy datasets with GEDI canopy data
to produce a panel dataset suitable for multivariate / fixed-effects regression:

$$Y_{it} = \beta_0 + \beta_1 C_{it} + \beta_2 X_{it} + \alpha_i + \gamma_t + \varepsilon_{it}$$

| Symbol | Meaning |
|---|---|
| $Y$ | Outcome: K-12 SOL pass rate, violent crime total, or health outcome % |
| $C$ | Canopy: `mean_canopy_cover` (GEDI L2B) or `canopy_height_mean_m` (GEDI L2A) |
| $X$ | Controls: median household income, population density, % bachelor's degree+ |
| $\alpha_i$ | Jurisdiction fixed effect |
| $\gamma_t$ | Year fixed effect |

**Data Sources:**
1. **GEDI L2A + L2B** — `s3://central-virginia-tree-canopy-project/dashboard-data/`
2. **VDOE SOL** — Virginia Open Data Portal + VDOE direct XLS (2019–2025)
3. **Virginia Crime** — FBI UCR Table 10 (2019) + VSP Annual Reports (2020–2024)
4. **CDC PLACES** — data.cdc.gov Socrata API (2024 release)
5. **Census ACS 5-Year** — api.census.gov (income, education, population density)

**Study Jurisdictions:** Albemarle, Augusta, Charlottesville, Fluvanna, Greene, Louisa, Nelson, Rockingham

**Study Years:** 2019–2025

## Cell 1 — Install Dependencies
Run once per kernel session. Restart the kernel after installation.

In [ ]:
import subprocess, sys

# Pin the boto stack to a mutually compatible set (required for SageMaker)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall",
    "aiobotocore==3.7.0",
    "botocore==1.43.0",
    "boto3==1.43.0",
    "s3fs==2026.6.0",
])

# Install all other dependencies
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "aioitertools",
    "openpyxl",
    "xlrd",
    "tabula-py",
    "linearmodels",
    "statsmodels",
    "tqdm",
])

print("All dependencies installed. Restart the kernel if this is the first run.")

## Cell 2 — Imports and Configuration

In [ ]:
import json
import logging
import os
import time
import warnings
from pathlib import Path

import boto3
import pandas as pd
import requests

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── S3 Configuration ──────────────────────────────────────────────────────────
S3_BUCKET          = "central-virginia-tree-canopy-project"
S3_DASHBOARD_PREFIX = "dashboard-data/"
GEDI_L2A_KEY       = "dashboard-data/gedi_county_summary.json"
GEDI_L2B_KEY       = "dashboard-data/gedi02B_county_summary.json"

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = Path("/tmp/policy_pipeline_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Census API key (set as SageMaker environment variable or paste here) ──────
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "")
if not CENSUS_API_KEY:
    log.warning(
        "CENSUS_API_KEY not set. ACS requests may be rate-limited.\n"
        "Get a free key at: https://api.census.gov/data/key_signup.html\n"
        "Set it with: import os; os.environ['CENSUS_API_KEY'] = 'your_key'"
    )

# ── Study-area constants ──────────────────────────────────────────────────────
DIVISION_CODES = {
    "Albemarle":       "002",
    "Augusta":         "015",
    "Charlottesville": "104",
    "Fluvanna":        "044",
    "Greene":          "055",
    "Louisa":          "075",
    "Nelson":          "090",
    "Rockingham":      "137",
}

FIPS_CODES = {
    "Albemarle":       "51003",
    "Augusta":         "51015",
    "Charlottesville": "51540",
    "Fluvanna":        "51065",
    "Greene":          "51079",
    "Louisa":          "51109",
    "Nelson":          "51125",
    "Rockingham":      "51165",
}

COUNTY_FIPS = {k: v[2:] for k, v in FIPS_CODES.items()}
STUDY_YEARS = list(range(2019, 2026))

print("Configuration loaded.")
print(f"Output directory: {OUTPUT_DIR}")
print(f"GEDI L2A: s3://{S3_BUCKET}/{GEDI_L2A_KEY}")
print(f"GEDI L2B: s3://{S3_BUCKET}/{GEDI_L2B_KEY}")

## Cell 3 — Helper: Download Utility

In [ ]:
def download(url: str, dest: Path, label: str, retries: int = 3) -> bool:
    """Download a URL to a local file. Returns True on success."""
    for attempt in range(1, retries + 1):
        try:
            log.info(f"[{label}] Downloading (attempt {attempt}): {url}")
            resp = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
            resp.raise_for_status()
            dest.write_bytes(resp.content)
            log.info(f"[{label}] Saved -> {dest}  ({len(resp.content):,} bytes)")
            return True
        except Exception as exc:
            log.warning(f"[{label}] Attempt {attempt} failed: {exc}")
            time.sleep(2 ** attempt)
    log.error(f"[{label}] All {retries} attempts failed.")
    return False

print("Helper functions defined.")

## Cell 4 — Load GEDI Data from S3

Loads `gedi_county_summary.json` (L2A — canopy height) and
`gedi02B_county_summary.json` (L2B — canopy cover) from
`s3://central-virginia-tree-canopy-project/dashboard-data/`
and merges them into a single GEDI panel.

In [ ]:
s3 = boto3.client("s3")

def load_json_from_s3(bucket: str, key: str, label: str) -> pd.DataFrame:
    """Download a JSON file from S3 and return it as a DataFrame."""
    local_path = OUTPUT_DIR / Path(key).name
    log.info(f"[{label}] Downloading s3://{bucket}/{key}")
    s3.download_file(bucket, key, str(local_path))
    data = json.loads(local_path.read_text())
    df = pd.DataFrame(data)
    log.info(f"[{label}] Loaded {len(df)} rows, columns: {list(df.columns)}")
    return df

# Load GEDI L2A (canopy height)
df_l2a = load_json_from_s3(S3_BUCKET, GEDI_L2A_KEY, "GEDI-L2A")

# Load GEDI L2B (canopy cover)
df_l2b = load_json_from_s3(S3_BUCKET, GEDI_L2B_KEY, "GEDI-L2B")

# Merge L2A and L2B on jurisdiction + year
l2a_cols = [c for c in ["jurisdiction", "year", "canopy_height_mean_m"] if c in df_l2a.columns]
l2b_cols = [c for c in ["jurisdiction", "year", "mean_canopy_cover", "total_valid_shots"] if c in df_l2b.columns]

gedi = pd.merge(
    df_l2a[l2a_cols],
    df_l2b[l2b_cols],
    on=["jurisdiction", "year"],
    how="outer",
)
gedi["jurisdiction"] = gedi["jurisdiction"].str.strip()
gedi["year"] = pd.to_numeric(gedi["year"], errors="coerce").astype("Int64")

print(f"\nGEDI panel: {len(gedi)} rows x {len(gedi.columns)} columns")
print(f"Jurisdictions: {sorted(gedi['jurisdiction'].unique())}")
print(f"Years: {sorted(gedi['year'].dropna().unique())}")
gedi.head(10)

## Cell 5 — Download VDOE SOL Pass Rates

Downloads Division Subject-Area XLS files from the VDOE website for school
years 2018-19 through 2024-25 and parses them into a tidy DataFrame.

In [ ]:
VDOE_XLS_URLS = {
    "2018-19": "https://www.doe.virginia.gov/home/showpublisheddocument/32592/638047216260530000",
    "2020-21": "https://www.doe.virginia.gov/home/showpublisheddocument/32568/638047216189130000",
    "2021-22": "https://www.doe.virginia.gov/home/showpublisheddocument/32594/638047216265870000",
    "2022-23": "https://www.doe.virginia.gov/home/showpublisheddocument/48939/638379725468370000",
    "2023-24": "https://www.doe.virginia.gov/home/showpublisheddocument/56492/638632943529970000",
    "2024-25": "https://www.doe.virginia.gov/home/showpublisheddocument/65231/638918222349300000",
}

NAME_TO_JURIS = {
    "albemarle county":       "Albemarle",
    "augusta county":         "Augusta",
    "charlottesville city":   "Charlottesville",
    "fluvanna county":        "Fluvanna",
    "greene county":          "Greene",
    "louisa county":          "Louisa",
    "nelson county":          "Nelson",
    "rockingham county":      "Rockingham",
}

def parse_vdoe_xls(path: Path, school_year_label: str) -> pd.DataFrame:
    """Parse a VDOE Division Subject-Area XLS into a tidy DataFrame."""
    try:
        xl = pd.read_excel(path, sheet_name=0, header=None)
    except Exception as exc:
        log.warning(f"[VDOE] Could not parse {path.name}: {exc}")
        return pd.DataFrame()

    # Find header row
    header_row = None
    for i, row in xl.iterrows():
        if any("division" in str(c).lower() for c in row.values):
            header_row = i
            break
    if header_row is None:
        log.warning(f"[VDOE] No header row found in {path.name}")
        return pd.DataFrame()

    xl.columns = (
        xl.iloc[header_row]
        .astype(str).str.strip().str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )
    xl = xl.iloc[header_row + 1:].reset_index(drop=True)

    div_col  = next((c for c in xl.columns if "division" in c), None)
    rate_col = next((c for c in xl.columns if "pass" in c and "rate" in c), None)
    subj_col = next((c for c in xl.columns if "subject" in c or "test" in c), None)

    if div_col is None or rate_col is None:
        log.warning(f"[VDOE] Missing required columns in {path.name}: {list(xl.columns)}")
        return pd.DataFrame()

    keep = [div_col, rate_col] + ([subj_col] if subj_col else [])
    xl = xl[keep].copy()
    xl.columns = ["division_name", "pass_rate"] + (["subject"] if subj_col else [])
    xl = xl.dropna(subset=["division_name", "pass_rate"])
    xl["pass_rate"] = pd.to_numeric(xl["pass_rate"], errors="coerce")

    if subj_col and "subject" in xl.columns:
        all_subj = xl[xl["subject"].astype(str).str.lower().str.contains("all", na=False)]
        if len(all_subj) == 0:
            xl = xl.groupby("division_name")["pass_rate"].mean().reset_index()
        else:
            xl = all_subj[["division_name", "pass_rate"]].copy()

    xl["division_name"] = xl["division_name"].astype(str).str.strip()
    xl["jurisdiction"] = xl["division_name"].str.lower().map(NAME_TO_JURIS)
    xl = xl.dropna(subset=["jurisdiction"])

    # Derive calendar year from school year label (e.g. "2022-23" -> 2023)
    yr_part = school_year_label.split("-")[1]
    cal_year = int(yr_part) + 2000 if len(yr_part) == 2 else int(yr_part)
    xl["year"] = cal_year

    return xl[["jurisdiction", "year", "pass_rate"]].rename(
        columns={"pass_rate": "sol_pass_rate_all"}
    )


sol_frames = []
for label, url in VDOE_XLS_URLS.items():
    dest = OUTPUT_DIR / f"vdoe_sol_{label.replace('-', '_')}.xlsx"
    ok = download(url, dest, f"VDOE/{label}")
    if ok:
        df = parse_vdoe_xls(dest, label)
        if not df.empty:
            log.info(f"[VDOE] {label}: {len(df)} rows parsed")
            sol_frames.append(df)

if sol_frames:
    sol = pd.concat(sol_frames, ignore_index=True)
    sol = sol.sort_values("year").drop_duplicates(subset=["jurisdiction", "year"], keep="last")
    print(f"\nVDOE SOL dataset: {len(sol)} rows")
    print(sol.pivot_table(index="jurisdiction", columns="year", values="sol_pass_rate_all"))
else:
    sol = pd.DataFrame(columns=["jurisdiction", "year", "sol_pass_rate_all"])
    print("WARNING: No VDOE SOL data retrieved. Check VDOE URL availability.")

## Cell 6 — Download Virginia Crime Data

Downloads FBI UCR Table 10 XLS (2019) and Virginia State Police annual
report PDFs (2020–2024). Attempts automatic PDF extraction via tabula-py.
If extraction fails, a manual instructions file is written to the output directory.

In [ ]:
FBI_UCR_2019_XLS = (
    "https://ucr.fbi.gov/crime-in-the-u.s/2019/crime-in-the-u.s.-2019"
    "/tables/table-10/table-10-state-cuts/virginia.xls"
)

VSP_PDF_URLS = {
    2020: "https://vsp.virginia.gov/wp-content/uploads/2022/06/Crime-In-Virginia-2020.pdf",
    2021: "https://vsp.virginia.gov/wp-content/uploads/2023/01/Crime-In-Virginia-2021.pdf",
    2022: "https://vsp.virginia.gov/wp-content/uploads/2024/01/Crime-In-Virginia-2022.pdf",
    2023: "https://vsp.virginia.gov/wp-content/uploads/2025/01/Crime-In-Virginia-2023.pdf",
    2024: "https://vsp.virginia.gov/wp-content/uploads/2026/03/Crime-In-Virginia-2024a.pdf",
}

CRIME_NAME_MAP = {
    "albemarle co.":          "Albemarle",
    "albemarle county":       "Albemarle",
    "augusta co.":            "Augusta",
    "augusta county":         "Augusta",
    "charlottesville":        "Charlottesville",
    "charlottesville city":   "Charlottesville",
    "fluvanna co.":           "Fluvanna",
    "fluvanna county":        "Fluvanna",
    "greene co.":             "Greene",
    "greene county":          "Greene",
    "louisa co.":             "Louisa",
    "louisa county":          "Louisa",
    "nelson co.":             "Nelson",
    "nelson county":          "Nelson",
    "rockingham co.":         "Rockingham",
    "rockingham county":      "Rockingham",
}

def parse_fbi_ucr_xls(path: Path, year: int) -> pd.DataFrame:
    try:
        xl = pd.read_excel(path, header=None, skiprows=3)
    except Exception as exc:
        log.warning(f"[Crime] Cannot parse {path.name}: {exc}")
        return pd.DataFrame()
    xl.columns = [f"c{i}" for i in range(xl.shape[1])]
    xl = xl.rename(columns={"c0": "county_name", "c2": "violent_crime_total"})
    xl = xl[["county_name", "violent_crime_total"]].dropna(subset=["county_name"])
    xl["county_name"] = xl["county_name"].astype(str).str.strip().str.lower()
    xl["jurisdiction"] = xl["county_name"].map(CRIME_NAME_MAP)
    xl = xl.dropna(subset=["jurisdiction"])
    xl["violent_crime_total"] = pd.to_numeric(xl["violent_crime_total"], errors="coerce")
    xl["year"] = year
    return xl[["jurisdiction", "year", "violent_crime_total"]]


def parse_vsp_pdf(path: Path, year: int) -> pd.DataFrame:
    try:
        import tabula
    except ImportError:
        log.info("[Crime] tabula-py not available — skipping PDF extraction.")
        return pd.DataFrame()
    try:
        tables = tabula.read_pdf(str(path), pages="all", multiple_tables=True,
                                 silent=True, pandas_options={"header": None})
        for tbl in tables:
            if tbl.shape[1] < 3:
                continue
            tbl.columns = [f"c{i}" for i in range(tbl.shape[1])]
            tbl["c0"] = tbl["c0"].astype(str).str.strip().str.lower()
            tbl["jurisdiction"] = tbl["c0"].map(CRIME_NAME_MAP)
            tbl = tbl.dropna(subset=["jurisdiction"])
            if len(tbl) >= 3:
                num_cols = [c for c in tbl.columns
                            if c not in ("c0", "jurisdiction")
                            and pd.to_numeric(tbl[c], errors="coerce").notna().sum() > 2]
                if num_cols:
                    tbl["violent_crime_total"] = pd.to_numeric(tbl[num_cols[0]], errors="coerce")
                    tbl["year"] = year
                    return tbl[["jurisdiction", "year", "violent_crime_total"]]
    except Exception as exc:
        log.warning(f"[Crime] tabula extraction failed for {path.name}: {exc}")
    return pd.DataFrame()


crime_frames = []

# 2019 — FBI UCR
dest_2019 = OUTPUT_DIR / "fbi_ucr_virginia_2019.xls"
if download(FBI_UCR_2019_XLS, dest_2019, "Crime/FBI-UCR-2019"):
    df = parse_fbi_ucr_xls(dest_2019, 2019)
    if not df.empty:
        crime_frames.append(df)
        log.info(f"[Crime] 2019 FBI UCR: {len(df)} rows")

# 2020-2024 — VSP PDFs
for yr, url in VSP_PDF_URLS.items():
    dest = OUTPUT_DIR / f"vsp_crime_virginia_{yr}.pdf"
    if download(url, dest, f"Crime/VSP-{yr}"):
        df = parse_vsp_pdf(dest, yr)
        if not df.empty:
            crime_frames.append(df)
            log.info(f"[Crime] VSP {yr}: {len(df)} rows parsed from PDF")
        else:
            log.info(f"[Crime] VSP {yr} PDF downloaded but not auto-parsed. "
                     "See crime_manual_instructions.txt for manual steps.")

if crime_frames:
    crime = pd.concat(crime_frames, ignore_index=True)
    crime = crime.drop_duplicates(subset=["jurisdiction", "year"], keep="last")
    print(f"\nCrime dataset: {len(crime)} rows")
    print(crime.pivot_table(index="jurisdiction", columns="year", values="violent_crime_total"))
else:
    crime = pd.DataFrame(columns=["jurisdiction", "year", "violent_crime_total"])
    manual_msg = (
        "MANUAL CRIME DATA STEPS\n"
        "=======================\n"
        "1. Go to: https://va.beyond2020.com/va_tops\n"
        "2. Select each jurisdiction and export years 2019-2024 as CSV.\n"
        "3. Save to: /tmp/policy_pipeline_output/crime_<jurisdiction>_<year>.csv\n"
        "4. Re-run this cell after placing the files.\n"
        "OR install tabula-py and re-run: pip install tabula-py\n"
    )
    (OUTPUT_DIR / "crime_manual_instructions.txt").write_text(manual_msg)
    print("WARNING: No crime data parsed. See crime_manual_instructions.txt")

## Cell 7 — Download CDC PLACES Health Data

Fetches county-level health estimates for all 8 study jurisdictions from
the CDC PLACES Socrata API (2024 release).

Key measures: obesity, diabetes, poor mental health, poor physical health, smoking.

In [ ]:
HEALTH_MEASURES = [
    "OBESITY", "DIABETES", "MHLTH", "PHLTH",
    "CSMOKING", "BPHIGH", "CASTHMA", "DEPRESSION",
]

fips_list = list(FIPS_CODES.values())
fips_filter = " OR ".join([f"countyfips='{f}'" for f in fips_list])

# Try 2024 release first, then fall back to alternate dataset ID
cdc_urls = [
    f"https://data.cdc.gov/resource/swc5-untb.json?$where={fips_filter}&$limit=500",
    f"https://data.cdc.gov/resource/fu4u-a9bh.json?$where={fips_filter}&$limit=500",
    f"https://data.cdc.gov/resource/d3i6-k6z5.json?$where={fips_filter}&$limit=500",
]

cdc_raw = []
for url in cdc_urls:
    dest = OUTPUT_DIR / "cdc_places_virginia.json"
    ok = download(url, dest, "CDC-PLACES")
    if ok:
        try:
            data = json.loads(dest.read_text())
            if data:
                cdc_raw = data
                log.info(f"[CDC] Retrieved {len(data)} records from {url}")
                break
        except Exception as exc:
            log.warning(f"[CDC] JSON parse failed: {exc}")

if cdc_raw:
    df_cdc = pd.DataFrame(cdc_raw)
    log.info(f"[CDC] Columns: {list(df_cdc.columns)[:12]}")

    fips_col  = next((c for c in df_cdc.columns if "fips" in c.lower()), None)
    meas_col  = next((c for c in df_cdc.columns if "measure" in c.lower()), None)
    val_col   = next((c for c in df_cdc.columns
                      if "crude" in c.lower() and "prev" in c.lower()), None)
    if val_col is None:
        val_col = next((c for c in df_cdc.columns if "data_value" in c.lower()), None)

    if fips_col and meas_col and val_col:
        df_cdc[val_col] = pd.to_numeric(df_cdc[val_col], errors="coerce")
        df_cdc["measure_short"] = df_cdc[meas_col].str.extract(
            r"(" + "|".join(HEALTH_MEASURES) + r")", expand=False
        )
        df_filtered = df_cdc[df_cdc["measure_short"].notna()].copy()

        cdc_pivot = df_filtered.pivot_table(
            index=fips_col, columns="measure_short",
            values=val_col, aggfunc="mean"
        ).reset_index()
        cdc_pivot.columns = [fips_col] + [
            f"health_{c.lower()}_pct" for c in cdc_pivot.columns[1:]
        ]

        fips_to_juris = {v: k for k, v in FIPS_CODES.items()}
        cdc_pivot["jurisdiction"] = cdc_pivot[fips_col].map(fips_to_juris)
        cdc = cdc_pivot.dropna(subset=["jurisdiction"]).drop(columns=[fips_col])
        cdc["places_data_year"] = 2024

        print(f"\nCDC PLACES dataset: {len(cdc)} jurisdictions")
        health_cols = [c for c in cdc.columns if "health_" in c]
        print(f"Health measures: {health_cols}")
        print(cdc[["jurisdiction"] + health_cols].to_string(index=False))
    else:
        log.warning(f"[CDC] Unexpected column structure. Saving raw for inspection.")
        df_cdc.to_csv(OUTPUT_DIR / "cdc_places_raw.csv", index=False)
        cdc = pd.DataFrame()
        print("WARNING: CDC PLACES column structure unexpected. Check cdc_places_raw.csv")
else:
    cdc = pd.DataFrame()
    print("WARNING: CDC PLACES data could not be retrieved.")

## Cell 8 — Download Census ACS Socioeconomic Controls

Fetches ACS 5-Year estimates for each study jurisdiction and year (2019–2024):
- Median household income (`B19013_001E`)
- Educational attainment — % bachelor's degree or higher
- Total population (for density calculation)

**Optional:** Set your Census API key to avoid rate limiting:
```python
import os; os.environ['CENSUS_API_KEY'] = 'your_key_here'
```
Free key at: https://api.census.gov/data/key_signup.html

In [ ]:
ACS_VARIABLES = [
    "B19013_001E",   # median household income
    "B15003_001E",   # population 25+ (education denominator)
    "B15003_022E",   # bachelor's degree
    "B15003_023E",   # master's degree
    "B15003_024E",   # professional degree
    "B15003_025E",   # doctorate degree
    "B01003_001E",   # total population
]

acs_frames = []
acs_years = [y for y in STUDY_YEARS if y <= 2024]

for year in acs_years:
    var_string = ",".join(["NAME"] + ACS_VARIABLES)
    base_url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = f"get={var_string}&for=county:*&in=state:51"
    if CENSUS_API_KEY:
        params += f"&key={CENSUS_API_KEY}"
    url = f"{base_url}?{params}"

    dest = OUTPUT_DIR / f"acs5_{year}_virginia.json"
    ok = download(url, dest, f"ACS/{year}")
    if not ok:
        continue

    try:
        raw = json.loads(dest.read_text())
    except Exception as exc:
        log.warning(f"[ACS] {year}: JSON parse failed: {exc}")
        continue

    if len(raw) < 2:
        log.warning(f"[ACS] {year}: Empty response")
        continue

    df = pd.DataFrame(raw[1:], columns=raw[0])
    df = df[df["county"].isin(COUNTY_FIPS.values())].copy()
    if df.empty:
        log.warning(f"[ACS] {year}: No matching counties found")
        continue

    county_to_juris = {v: k for k, v in COUNTY_FIPS.items()}
    df["jurisdiction"] = df["county"].map(county_to_juris)
    df["year"] = year

    for col in ACS_VARIABLES:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["median_household_income"] = df["B19013_001E"]
    df["total_population"]        = df["B01003_001E"]
    df["pct_bachelors_plus"] = (
        (df["B15003_022E"] + df["B15003_023E"] +
         df["B15003_024E"] + df["B15003_025E"])
        / df["B15003_001E"] * 100
    ).round(2)

    acs_frames.append(df[["jurisdiction", "year", "median_household_income",
                           "total_population", "pct_bachelors_plus"]])
    log.info(f"[ACS] {year}: {len(df)} jurisdictions loaded")

if acs_frames:
    acs = pd.concat(acs_frames, ignore_index=True)

    # Add population density using TIGER land area
    tiger_url = "https://api.census.gov/data/2020/dec/pl?get=NAME,AREALAND&for=county:*&in=state:51"
    if CENSUS_API_KEY:
        tiger_url += f"&key={CENSUS_API_KEY}"
    dest_tiger = OUTPUT_DIR / "tiger_land_area_virginia.json"
    if download(tiger_url, dest_tiger, "TIGER/LandArea"):
        try:
            raw_tiger = json.loads(dest_tiger.read_text())
            land = pd.DataFrame(raw_tiger[1:], columns=raw_tiger[0])
            land = land[land["county"].isin(COUNTY_FIPS.values())].copy()
            county_to_juris = {v: k for k, v in COUNTY_FIPS.items()}
            land["jurisdiction"] = land["county"].map(county_to_juris)
            land["land_area_sqmi"] = pd.to_numeric(land["AREALAND"], errors="coerce") / 2_589_988.11
            acs = acs.merge(land[["jurisdiction", "land_area_sqmi"]], on="jurisdiction", how="left")
            acs["pop_density_per_sqmi"] = (acs["total_population"] / acs["land_area_sqmi"]).round(2)
            acs = acs.drop(columns=["land_area_sqmi"])
            log.info("[ACS] Population density calculated.")
        except Exception as exc:
            log.warning(f"[ACS] Density calculation failed: {exc}")
            acs["pop_density_per_sqmi"] = float("nan")
    else:
        acs["pop_density_per_sqmi"] = float("nan")

    print(f"\nACS dataset: {len(acs)} rows")
    print(acs.pivot_table(index="jurisdiction", columns="year", values="median_household_income"))
else:
    acs = pd.DataFrame(columns=["jurisdiction", "year", "median_household_income",
                                 "total_population", "pct_bachelors_plus", "pop_density_per_sqmi"])
    print("WARNING: No ACS data retrieved. Set CENSUS_API_KEY and re-run.")

## Cell 9 — Merge All Datasets into the Panel

Merges GEDI (spine) with SOL, crime, CDC PLACES, and ACS on `(jurisdiction, year)`.
CDC PLACES is cross-sectional and is broadcast across all years.

In [ ]:
panel = gedi.copy()

# SOL pass rates
if not sol.empty:
    panel = panel.merge(sol, on=["jurisdiction", "year"], how="left")
    log.info(f"After SOL join: {len(panel)} rows, "
             f"{panel['sol_pass_rate_all'].notna().sum()} SOL values")

# Crime data
if not crime.empty:
    panel = panel.merge(crime, on=["jurisdiction", "year"], how="left")
    log.info(f"After crime join: {len(panel)} rows")

# ACS controls
if not acs.empty:
    panel = panel.merge(acs, on=["jurisdiction", "year"], how="left")
    log.info(f"After ACS join: {len(panel)} rows")

# CDC PLACES (cross-sectional — broadcast across all years)
if not cdc.empty:
    panel = panel.merge(cdc, on="jurisdiction", how="left")
    log.info(f"After CDC PLACES join: {len(panel)} rows")

# Add FIPS codes
panel["fips"] = panel["jurisdiction"].map(FIPS_CODES)

# Data quality flags
panel["flag_low_gedi_shots"] = (
    panel["total_valid_shots"].notna() & (panel["total_valid_shots"] < 1000)
)
if "sol_pass_rate_all" in panel.columns:
    panel["flag_sol_missing"] = panel["sol_pass_rate_all"].isna()
if "violent_crime_total" in panel.columns:
    panel["flag_crime_missing"] = panel["violent_crime_total"].isna()

panel = panel.sort_values(["jurisdiction", "year"]).reset_index(drop=True)

print(f"\nFinal panel dataset: {len(panel)} rows x {len(panel.columns)} columns")
print(f"Jurisdictions: {sorted(panel['jurisdiction'].unique())}")
print(f"Years: {sorted(panel['year'].dropna().unique())}")
print(f"Columns: {list(panel.columns)}")
panel.head(16)

## Cell 10 — Data Coverage Summary

In [ ]:
import pandas as pd

outcome_cols = [
    "mean_canopy_cover",
    "canopy_height_mean_m",
    "sol_pass_rate_all",
    "violent_crime_total",
    "median_household_income",
    "pct_bachelors_plus",
    "pop_density_per_sqmi",
]
outcome_cols = [c for c in outcome_cols if c in panel.columns]

coverage = pd.DataFrame({
    "column": outcome_cols,
    "non_null": [panel[c].notna().sum() for c in outcome_cols],
    "total": len(panel),
})
coverage["coverage_pct"] = (coverage["non_null"] / coverage["total"] * 100).round(1)

print("Data Coverage Summary")
print("=" * 50)
print(coverage.to_string(index=False))

# Also show low-GEDI-shot flags
if "flag_low_gedi_shots" in panel.columns:
    low_shot_rows = panel[panel["flag_low_gedi_shots"] == True][["jurisdiction", "year", "total_valid_shots"]]
    if not low_shot_rows.empty:
        print(f"\nLow GEDI shot count rows (< 1000 shots):")
        print(low_shot_rows.to_string(index=False))

## Cell 11 — Save Outputs to Local and S3

Saves the panel dataset to `/tmp/policy_pipeline_output/` and uploads
to `s3://central-virginia-tree-canopy-project/dashboard-data/`.

In [ ]:
S3_OUTPUT_DESTINATIONS = [
    ("central-virginia-tree-canopy-project", "dashboard-data/"),
    ("central-va-tree-canopy-dashboard",     "dashboard-data/"),
    ("central-va-tree-canopy-dashboard",     "data/"),
]

def upload_to_s3_all(local_path: Path, filename: str, label: str) -> None:
    """Upload a file to all three S3 destinations."""
    for bucket, prefix in S3_OUTPUT_DESTINATIONS:
        key = prefix + filename
        try:
            s3.upload_file(str(local_path), bucket, key)
            print(f"  -> s3://{bucket}/{key}")
        except Exception as exc:
            log.warning(f"  [WARN] Failed to upload to s3://{bucket}/{key}: {exc}")

# Save long-format panel CSV
long_path = OUTPUT_DIR / "policy_panel_dataset.csv"
panel.to_csv(long_path, index=False)
print(f"Saved locally: {long_path}")
print("Uploading policy_panel_dataset.csv to S3...")
upload_to_s3_all(long_path, "policy_panel_dataset.csv", "Panel CSV")

# Save wide-format panel CSV
wide_cols = [c for c in ["mean_canopy_cover", "canopy_height_mean_m",
                          "sol_pass_rate_all", "median_household_income"]
             if c in panel.columns]
if wide_cols:
    wide = panel.pivot_table(
        index="jurisdiction", columns="year", values=wide_cols, aggfunc="mean"
    )
    wide.columns = [f"{v}_{y}" for v, y in wide.columns]
    wide_path = OUTPUT_DIR / "policy_panel_dataset_wide.csv"
    wide.reset_index().to_csv(wide_path, index=False)
    print(f"\nSaved wide-format locally: {wide_path}")
    print("Uploading policy_panel_dataset_wide.csv to S3...")
    upload_to_s3_all(wide_path, "policy_panel_dataset_wide.csv", "Panel Wide CSV")

# Save panel as JSON for dashboard use
json_path = OUTPUT_DIR / "policy_panel_dataset.json"
panel.astype(object).where(panel.notna(), other=None).to_json(
    json_path, orient="records", indent=2
)
print(f"\nSaved JSON locally: {json_path}")
print("Uploading policy_panel_dataset.json to S3...")
upload_to_s3_all(json_path, "policy_panel_dataset.json", "Panel JSON")

print("\nAll outputs saved.")

## Cell 12 — Fixed Effects Panel Regression

Runs a two-way Fixed Effects regression for each combination of:
- **Canopy variable:** `mean_canopy_cover`, `canopy_height_mean_m`
- **Outcome variable:** SOL pass rate, violent crime, obesity, diabetes, poor mental health

**Model:**
$$Y_{it} = \beta_1 C_{it} + \beta_2 X_{it} + \alpha_i + \gamma_t + \varepsilon_{it}$$

Requires: `pip install linearmodels statsmodels` (installed in Cell 1)

In [ ]:
try:
    from linearmodels.panel import PanelOLS
    import statsmodels.api as sm
    HAS_LINEARMODELS = True
except ImportError:
    HAS_LINEARMODELS = False
    print("linearmodels not available. Run Cell 1 and restart the kernel.")

if HAS_LINEARMODELS:
    df_reg = panel.copy()
    df_reg = df_reg.set_index(["jurisdiction", "year"])

    canopy_vars  = [c for c in ["mean_canopy_cover", "canopy_height_mean_m"] if c in df_reg.columns]
    control_vars = [c for c in ["median_household_income", "pop_density_per_sqmi",
                                 "pct_bachelors_plus"] if c in df_reg.columns]
    outcome_vars = {
        "sol_pass_rate_all":    "K-12 SOL Pass Rate (%)",
        "violent_crime_total":  "Violent Crime Total",
        "health_obesity_pct":   "Obesity Prevalence (%)",
        "health_diabetes_pct":  "Diabetes Prevalence (%)",
        "health_mhlth_pct":     "Poor Mental Health (%)",
    }
    outcome_vars = {k: v for k, v in outcome_vars.items() if k in df_reg.columns}

    results_rows = []

    for canopy_var in canopy_vars:
        for outcome_var, outcome_label in outcome_vars.items():
            req = [outcome_var, canopy_var] + control_vars
            df_m = df_reg[req].dropna()

            if len(df_m) < 10:
                log.warning(f"Insufficient data: {outcome_var} ~ {canopy_var} (n={len(df_m)})")
                continue

            y = df_m[outcome_var]
            X = sm.add_constant(df_m[[canopy_var] + control_vars])

            try:
                model  = PanelOLS(y, X, entity_effects=True, time_effects=True)
                result = model.fit(cov_type="clustered", cluster_entity=True)

                coef = result.params.get(canopy_var, float("nan"))
                pval = result.pvalues.get(canopy_var, float("nan"))
                r2   = result.rsquared

                results_rows.append({
                    "outcome":     outcome_label,
                    "canopy_var":  canopy_var,
                    "beta":        round(coef, 4),
                    "p_value":     round(pval, 4),
                    "r_squared":   round(r2, 3),
                    "n_obs":       len(df_m),
                    "significant": pval < 0.05,
                })
            except Exception as exc:
                log.warning(f"Model failed: {outcome_var} ~ {canopy_var}: {exc}")

    if results_rows:
        results_df = pd.DataFrame(results_rows)

        # Save locally and upload to S3
        reg_path = OUTPUT_DIR / "regression_results.csv"
        results_df.to_csv(reg_path, index=False)
        upload_to_s3_all(reg_path, "regression_results.csv", "Regression Results")

        print("\n" + "=" * 70)
        print("FIXED EFFECTS REGRESSION RESULTS")
        print("Model: Y_it = beta1*C_it + beta2*X_it + alpha_i + gamma_t + eps_it")
        print("=" * 70)
        print(results_df.to_string(index=False))
        print("=" * 70)

        sig = results_df[results_df["significant"] == True]
        if not sig.empty:
            print(f"\nStatistically significant findings (p < 0.05):")
            print(sig[["outcome", "canopy_var", "beta", "p_value", "n_obs"]].to_string(index=False))
        else:
            print("\nNo statistically significant findings at p < 0.05.")
            print("Note: With N=8 jurisdictions and T<=7 years, statistical power is limited.")
            print("Consider this a pilot study requiring additional data collection.")
    else:
        print("No regression models could be estimated. Check data coverage in Cell 10.")

## Notes and Limitations

### Data Quality Flags
- **2019 SMAP anomaly:** The 2019 SMAP soil moisture value covers only 32 days (Jan–Feb). Treat with caution in any temporal analysis.
- **Charlottesville GEDI shots:** Charlottesville City has very low GEDI shot counts in some years (as few as 21 shots in 2025) due to its small geographic footprint. Rows flagged with `flag_low_gedi_shots = True` should be excluded from regression models or treated as outliers.
- **2023 canopy cover drop:** All jurisdictions show a sharp drop in `mean_canopy_cover` in 2023. This may reflect a satellite coverage gap, drought stress, or real deforestation. Investigate before including 2023 in regression models.

### Statistical Power
With N=8 jurisdictions and T=5–7 years, the panel has at most 56 observations. After removing missing values and applying fixed effects, the effective degrees of freedom are very limited. The regression results should be interpreted as **exploratory/hypothesis-generating**, not as definitive causal evidence.

### Crime Data
If automatic PDF extraction failed, follow the instructions in `crime_manual_instructions.txt` to manually download the data from [va.beyond2020.com](https://va.beyond2020.com/va_tops).

### CDC PLACES
CDC PLACES county data is cross-sectional (one estimate per county, not per year). It is broadcast across all study years in this pipeline. For longitudinal health analysis, consider the CDC PLACES trend data or BRFSS county-level estimates.